In [1]:
import pandas as pd
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Segment, Timeline, Annotation
import os

def convert_model_csv_to_rrtm(model_root_folder, csv_filename):
    """
    Convert the diarization model's CSV output to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        model_root_folder (str): The root folder where the model's CSV output is located.
        csv_filename (str): The name of the CSV file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the CSV file in the diarization model's output folder.
    """
    with open(os.path.join(model_root_folder, csv_filename[:-4]+".rttm"), 'w') as rttm_file:
        df = pd.read_csv(f"{model_root_folder}/{csv_filename}")
        for _, row in df.iterrows():
            if(row['speaker'] == 'speaker_0'):
                begin = float(row['start_time'])
                end = float(row['end_time'])
                speaker = row['speaker']
                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                rttm_file.write(rttm_line)

def convert_gold_label_txt_to_rrtm(gold_label_root_folder, txt_filename):
    """
    Convert the gold label diarization annotations from a TXT file (from ELAN) to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        gold_label_root_folder (str): The root folder where the gold label TXT file is located.
        txt_filename (str): The name of the TXT file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the TXT file in the gold label annotations folder.
    """

    df = pd.read_csv(os.path.join(gold_label_root_folder, txt_filename), sep='\t')
    # Create RTTM output
    with open(os.path.join(gold_label_root_folder, txt_filename[:-4]+".rttm"), 'w') as rttm_file:
        for idx, row in df.iterrows():
            if(row['speaker'] == 'SPEAKER_1'):
                begin = float(row['speech_start'])
                end = float(row['speech_end'])
                speaker = row['speaker']
                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                rttm_file.write(rttm_line)

def load_rttm_to_annotation(path):
    """Load an RTTM file and convert it to a pyannote.core.Annotation object.
    Args:
        path (str): Path to the RTTM file.
    Returns:
        pyannote.core.Annotation: The loaded annotation."""
    ann = Annotation()
    with open(path, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] != 'SPEAKER':
                continue
            # handle both custom and standard RTTM token positions
            try:
                start = float(parts[1])
                dur = float(parts[2])
                speaker = parts[-1]
            except Exception:
                start = float(parts[3])
                dur = float(parts[4])
                speaker = parts[-1]
            ann[Segment(start, start + dur)] = speaker
    return ann

def der_component_conversion(der_dict):
    """Convert the DER components from a dictionary to a more human-readable format.
    Args:
        der_dict (dict): The dictionary containing DER components.
    Returns:
        dict: A dictionary with human-readable DER components."""
    return {
        'missed_speech_percent': f"{100*der_dict['missed detection']/der_dict['total']:.2f}",
        'false_alarm_percent': f"{100*der_dict['false alarm']/der_dict['total']:.2f}",
        'confusion_percent': f"{100*der_dict['confusion']/der_dict['total']:.2f}",
        "correct_percent": f"{100*der_dict['correct']/der_dict['total']:.2f}",
        "der": f"{100*der_dict['diarization error rate']:.2f}"
    }

In [2]:
ref_root = "gold_label_diarization_annotations"
hyp_root = "outputs/"
df_overlap = {"id": [], "MSDD?": [], "DER": [], "False Alarm %": [], "Confusion %": [], "Missed Speech Detection %": [], "Correct %": []} 
ids = ["p9-s9"] #, "p7-s8", "p9-s9", "p11-s4", "p11-s11", "p12-s3", "p12-s6", "p17-s2", "p18-s15"]

for id in ids:
    ref_file_name = f"{id}_gold_labels"
    hyp_file_name = f"{id}_denoised"

    convert_model_csv_to_rrtm(hyp_root, hyp_file_name + ".csv")
    convert_gold_label_txt_to_rrtm(ref_root, ref_file_name + ".txt")

    ref = load_rttm_to_annotation(os.path.join(ref_root, ref_file_name+'.rttm'))
    hyp = load_rttm_to_annotation(os.path.join(hyp_root, hyp_file_name+'.rttm'))

    # consider_overlap = False
    # metric = DiarizationErrorRate(collar = 0.25, skip_overlap=(not consider_overlap))
    # der = der_component_conversion(metric(ref, hyp, detailed=True))
    # #print(f"Diarization Error Rate: {der_component_conversion(der)} (considering overlap = {consider_overlap})")
    # df_overlap["id"].append(id)
    # df_overlap["considering overlap?"].append("No")
    # df_overlap["DER"].append(der["der"])
    # df_overlap["False Alarm %"].append(der["false_alarm_percent"])
    # df_overlap["Confusion %"].append(der["confusion_percent"])
    # df_overlap["Missed Speech Detection %"].append(der["missed_speech_percent"])
    # df_overlap["Correct %"].append(der["correct_percent"])

    # metric = DiarizationErrorRate(collar = 0.25, skip_overlap=(consider_overlap))
    # der = der_component_conversion(metric(ref, hyp, detailed=True))
    # #print(f"Diarization Error Rate: {der_component_conversion(der)} (considering overlap = {consider_overlap})")
    # df_overlap["id"].append(id)
    # df_overlap["considering overlap?"].append("Yes")
    # df_overlap["DER"].append(der["der"])
    # df_overlap["False Alarm %"].append(der["false_alarm_percent"])
    # df_overlap["Confusion %"].append(der["confusion_percent"])
    # df_overlap["Missed Speech Detection %"].append(der["missed_speech_percent"])
    # df_overlap["Correct %"].append(der["correct_percent"])

    metric = DiarizationErrorRate(collar = 0.25, skip_overlap=False)
    der = der_component_conversion(metric(ref, hyp, detailed=True))
    #print(f"Diarization Error Rate: {der_component_conversion(der)} (considering overlap = {consider_overlap})")
    df_overlap["id"].append(id)
    df_overlap["MSDD?"].append("Yes")
    df_overlap["DER"].append(der["der"])
    df_overlap["False Alarm %"].append(der["false_alarm_percent"])
    df_overlap["Confusion %"].append(der["confusion_percent"])
    df_overlap["Missed Speech Detection %"].append(der["missed_speech_percent"])
    df_overlap["Correct %"].append(der["correct_percent"])

df_overlap = pd.DataFrame(df_overlap)

e:\Vertibri\Project\Testing_folder\der_env\Lib\site-packages\pyannote\metrics\utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


In [3]:
df_overlap.style.hide()
df_overlap[["id", "MSDD?", "DER", "False Alarm %", "Missed Speech Detection %", "Confusion %", "Correct %"]].style.hide()

id,MSDD?,DER,False Alarm %,Missed Speech Detection %,Confusion %,Correct %
p9-s9,Yes,61.76,3.43,58.32,0.00,41.68
